<h1> Notebook Summary </h1>

Partially adapted from : ...
TODO: Add a summary of the notebook here. Also include instructions on creating a venv

<h1> Retrieving the GEDI data from NASA Earthdata </h1>

<h3> We first need to import required dependencies </h3>

In [ ]:
from harmony import BBox, Client, Collection, Request, CapabilitiesRequest # NASA Harmony
import earthaccess # NASA Earthdata Login
from datetime import datetime
import os
import json
import requests
import tempfile
import h5py
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import box

<h3> Login to NASA Earthdata </h3>

You may need to register for a profile on NASA Earthdata to do this. Here is the link: https://urs.earthdata.nasa.gov/users/new

Once you are logged in, click on "Generate Token" to create an API token to allow you to access Earthdata. Save this token somewhere safe. 

In [5]:
auth = earthaccess.login(persist=True)

<h3> Upload GeoJSON of area of interest </h3>

To retrieve the area of interest, we will need a GeoJSON file containing a bounding box of the area of interest. Once you have this file, store it **in the same folder** as this notebook. 

You may only have **ONE GeoJSON file in this folder**, and it **must be named with the following convention: [place_name].geojson.**

In [ ]:
curr_dir = os.getcwd()
geojson_files = [f for f in os.listdir(curr_dir) if f.endswith('.geojson')]

if len(geojson_files) == 0:
    print("No GeoJSON file found in the current directory. Please add a GeoJSON file and try again.")
elif len(geojson_files) > 1:
    print("Multiple GeoJSON files found in the current directory. Please ensure there is only one GeoJSON file and try again.")
else:
    geojson_file_path = os.path.join(curr_dir, geojson_files[0])
    try:
        with open(geojson_file_path) as f:
            geojson_polygon = json.load(f)
        print("GeoJSON loaded successfully: ", geojson_file_path)
    except json.JSONDecodeError:
        print(f"Invalid GeoJSON format in file: {geojson_file_path}")

<h3> Select the dates for retreival </h3>

GEDI data is available during the following time periods:
*   April 2019 - March 2023
*   April 2024 - Present

If doing multiple retrievals, it is recommended that you don't do a retrrieval for more than 730 days/2 years at a time to ensure all data is retrieved successfully and to avoid timeouts.

In [ ]:
start_str = input("Enter start date (YYYY-MM-DD): ")
stop_str = input("Enter stop date (YYYY-MM-DD): ")

# Convert to datetime 
try:
    temporal_range = {
        'start': datetime.strptime(start_str, "%Y-%m-%d"),
        'stop': datetime.strptime(stop_str, "%Y-%m-%d")
    }
    print("Start date: ", temporal_range['start'])
    print("Stop date: ", temporal_range['stop'])
except ValueError:
    print("Invalid date format. Please try again and use YYYY-MM-DD.")

<h3> Call Harmony to retrieve the GEDI data h5 files</h3>

**NOTE: ** It is likely that your request will pause after a bit (unless it is a very small date range or area of interest). To resume, you will need to run the chunk below the following chunk containing:

`task_json = harmony_client.resume(task)`
`task_json = harmony_client.result_json(task, show_progress=True)`


In [ ]:
harmony_client = Client(auth=(auth.username, auth.password))
capabilities_request = CapabilitiesRequest(short_name='GEDI02_A')
capabilities = harmony_client.submit(capabilities_request)
concept_id = capabilities['conceptId']

request = Request(
    collection = Collection(id=concept_id),
    shape = geojson_file_path,
    temporal = temporal_range
)
task = harmony_client.submit(request)
print(f'Harmony request ID: {task}')
print(f'Processing your Harmony request:')

task_json = harmony_client.result_json(task, show_progress=True)

In [ ]:
# Run this if the job is paused by harmony
task_json = harmony_client.resume(task)
task_json = harmony_client.result_json(task, show_progress=True)

<h3> Save the retrieved h5 files in a txt file </h3>

The h5 files contain api links to the actual data. We will save these api links in a txt file to make it easier to download the data using a script. The h5 files will be saved in the same folder as this notebook. 

In [ ]:
h5_files = [link['href'] for link in task_json['links'] if link['href'].endswith('.h5')]
with open(os.path.join(os.getcwd(), 'h5_files.txt'), 'w') as f:
    for h5_file in h5_files:
        f.write(h5_file + '\n')

print("H5 file links saved to h5_files.txt")

<h3> Ask about quality and other filtering options </h3>

Sensitivity: This is a measure of the quality/accuracy of the data. The higher your threshold, the more accurate your data will be, but you will also get less data. It's recommended to set your threshold between 0.5 and 0.9. 


RH95: This is the 95th percentile of tree heights in a specific area that the GEDI data is collected from (since every point is ~20m in diameter). Setting a rh95 threshold filters out data that may be noisy. For instance, having negative rh95 values is not possible as trees cant be negative heights, so setting a rh95 threshold of 0 or higher can help filter out noisy data. Setting a rh95 threshold of 2 or higher can help filter out data that may be from areas with very short vegetation (such as grasslands).

Nighttime Data: This is data that is collected at night. The benefit of using data from only the night is that it is not affected by solar noise at similar wavelengths. However, though much more accurate, this will get rid of a lot of data as it only keeps points with solar elevation < 0.

In [ ]:
min_sensitivity_threshold = float(input("Enter the minimum sensitivity threshold (recommended between 0.5 and 0.9): "))
min_rh95_threshold = float(input("Enter the minimum rh95 threshold (decimal between 0 and 5): "))
nighttime_data_only = input("Do you want to include only nighttime data? (Y/N): ")

nighttime_data_only = bool(nighttime_data_only.strip().upper() == 'Y')

if min_sensitivity_threshold < 0 or min_sensitivity_threshold > 1:
    print("Invalid sensitivity threshold. Please re-run and enter a decimal value between 0 and 1.")
    exit(1)

print(f"Minimum sensitivity threshold set to: {min_sensitivity_threshold}")
print(f"Minimum rh95 threshold set to: {min_rh95_threshold}")
print(f"Nighttime data only: {nighttime_data_only}")

<h3> Create function to filter the data based on the above options </h3>

Adopted from ____________DO HEREE________

In [ ]:
def extract_gedi_rh_metrics_from_urls(h5_files_list, output_parquet_file, beams=None,
                                      min_sensitivity=0.9, min_rh95=1,
                                      night=False):
    """
    Reads a list of HTTPS GEDI HDF5 file URLs, downloads each file completely,
    extracts RH metrics for specified beams, filters by sensitivity, RH95,
    and optionally by solar elevation (nighttime), and saves all shots to a parquet file.

    Parameters
    ----------
    h5_files_list : str
        Path to a text file containing one HTTPS URL per line.
    output_parquet_file : str
        Output parquet file path.
    beams : list, optional
        List of beam names to extract. Defaults to the four full-power beams.
    min_sensitivity : float, optional
        Minimum acceptable sensitivity (default: 0.95)
    min_rh95 : float, optional
        Minimum acceptable RH95 value (default: 0)
    night : bool, optional
        If True, select only shots where solar_elevation < 0 (nighttime)
    """
    if beams is None:
        beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']

    # Read list of URLs
    with open(h5_files_list, 'r') as f:
        h5_urls = [line.strip() for line in f if line.strip()]

    records = []

    for url in h5_urls:
        print(f"\n📥 Downloading {url} ...")
        tmp_path = None

        try:
            response = requests.get(url, timeout=300) # works because earthaccess.login(persist=True) stored creds in ~/.netrc
            response.raise_for_status()
            
            # save file to temp directory
            with tempfile.NamedTemporaryFile(suffix=".h5", delete=False) as tmp:
                tmp.write(response.content)
                tmp_path = tmp.name

            with h5py.File(tmp_path, 'r') as f:
                for beam_name in beams:
                    if beam_name not in f:
                        print(f"  ⚠️ Beam {beam_name} not found in {url}")
                        continue

                    beam = f[beam_name]

                    # Check datasets required for GEDI L2A v2.1+
                    required = ['lat_lowestmode', 'lon_lowestmode', 'rh', 'shot_number', 'solar_elevation']
                    missing = [k for k in required if k not in beam]
                    if len(missing) > 0:
                        print(f"  ⚠️ Missing the following data for {beam_name} in {url}: \n{missing}")
                        continue

                    # Try to access sensitivity safely
                    sensitivity = None
                    if 'QA' in beam and 'sensitivity' in beam['QA']:
                        sensitivity = beam['QA/sensitivity'][:]
                    elif 'sensitivity' in beam:
                        sensitivity = beam['sensitivity'][:]
                    else:
                        print(f"  ⚠️ Sensitivity missing in {beam_name} in {url}, skipping beam")
                        continue

                    lat = beam['lat_lowestmode'][:]
                    lon = beam['lon_lowestmode'][:]
                    shot_num = beam['shot_number'][:]
                    rh = beam['rh'][:]
                    solar = beam['solar_elevation'][:]

                    # Extract selected RH metrics
                    rh25 = rh[:, 25]
                    rh50 = rh[:, 50]
                    rh75 = rh[:, 75]
                    rh95 = rh[:, 95]

                    # Apply filters
                    mask = (sensitivity >= min_sensitivity) & (rh95 >= min_rh95)
                    if night:
                        mask &= (solar < 0)

                    if not mask.any():
                        print(f"  NOTE: ⚙️ No shots passed filters in {beam_name} in {url}")
                        continue

                    df = pd.DataFrame({
                        "source_url": url,
                        "beam": beam_name,
                        "shot_number": shot_num[mask],
                        "latitude": lat[mask],
                        "longitude": lon[mask],
                        "rh25": rh25[mask],
                        "rh50": rh50[mask],
                        "rh75": rh75[mask],
                        "rh95": rh95[mask],
                        "sensitivity": sensitivity[mask],
                        "solar_elevation": solar[mask]
                    })

                    records.append(df)

        except requests.exceptions.RequestException as e:
            print(f"❌ Download failed for {url}: {e}")
        except Exception as e:
            print(f"❌ Error processing {url}: {e}")
        finally:
            if 'tmp_path' in locals() and os.path.exists(tmp_path):
                os.remove(tmp_path)

    # Combine all dataframes
    if records:
        shots_df = pd.concat(records, ignore_index=True)

        shots_geodf = gpd.GeoDataFrame(
            shots_df,
            geometry=gpd.points_from_xy(shots_df["longitude"], shots_df["latitude"]),
            crs="EPSG:4326"
        )

        shots_geodf.to_parquet(output_parquet_file,  
                                        compression='snappy')

        print(f"\n✅ Total shots saved: {len(shots_df):,}")
        print(f"✅ Output file: {output_parquet_file}")
    else:
        print("\n⚠️ No valid shots extracted.")

<h3> [Optional] Select the output folder for your GEDI results </h3>

If you do not want the outputted GeoParquet of GEDI data to be stored in the same folder as this notebook, you can specify an output folder here. If you do not specify an output folder, the GeoParquet file will be stored in the same folder as this notebook.

The output folder should be FROM THE ROOT of this repository. In VSCode, you can right click on the desired output folder and select "Copy Relative Path" to get the correct path.

If you keep recieving errors, just do not specify an output folder and the results will be saved in the same folder as this notebook. This is the simplest.

In [ ]:
output_folder = input("Enter the path to the output folder where you wish to save your results, or press Enter to use the current folder:")

if output_folder.strip() == "":
    output_folder = curr_dir
else:
    if not os.path.exists(output_folder):
        print(f"Output folder '{output_folder}' does not exist. Please create the folder and try again.")
        exit(1)

print(f"GEDI data will be saved in GeoParquet format in: {output_folder}")

<h3> Run the above function on the retrieved h5 files with the specified filtering options and save the results in a GeoParquet File </h3>

In [ ]:
h5_files = os.path.join(os.getcwd(), 'h5_files.txt')
out_parquet_file = os.path.join(output_folder, 'gedi_shots.parquet')

#create the file first
try:
    with open(out_parquet_file, 'w') as f:
        f.write('') # just create an empty file
except Exception as e:
    with open(os.path.join(curr_dir, 'gedi_shots.parquet'), 'w') as f:
        f.write('')
    out_parquet_file = os.path.join(curr_dir, 'gedi_shots.parquet')
    print(f"WARNING: Could not create output file in specified folder. Defaulting to current directory. Error details: {e}")

#List of Full Power Beams
full_power_beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']

#Run the function to filter H5 file and extract RH metrics
extract_gedi_rh_metrics_from_urls(h5_files, out_parquet_file, 
                                  beams=full_power_beams,
                                  min_sensitivity=min_sensitivity_threshold, 
                                  min_rh95=min_rh95_threshold,
                                  night=nighttime_data_only)

<h3> Function To Create Grid Tiles </h3>

This function splits up the area of interest into a square grid of tiles. Then it looks at the GEDI shots that fall within each tile, and saves those shots to a parquet file. It also saves the grid itself as a parquet file.

In [ ]:
# This is to get dimensions of the GeoJSON
def tile_gedi_points_by_aoi(
    shots_gdf : gpd.GeoDataFrame,
    aoi_gdf : gpd.GeoDataFrame,
    output_dir : str = "tiled_gedi_points",
    prefix : str = "gedi",
    grid_size : int = 10,
    preferred_utm : str = None, # NOTE: FOR PARA, use SIRGAS 2000 / Brazil Albers (EPSG:10857) 
):
    # create output directory if it don't alr exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # give CRS if not already set 
    original_shots_crs = shots_gdf.crs
    if original_shots_crs is None:
        shots_gdf = shots_gdf.set_crs("EPSG:4326")
    original_aoi_crs = aoi_gdf.crs
    if original_aoi_crs is None:
        aoi_gdf = aoi_gdf.set_crs("EPSG:4326")

    # lat-long isn't equal across the globe, so convert to a projected CRS
    if preferred_utm:
        grid_crs = preferred_utm
        aoi_proj = aoi_gdf.to_crs(preferred_utm)
        shots_proj = shots_gdf.to_crs(preferred_utm)
    else:
        try:
            # ask gpd to estimate a suitable meter CRS for the AOI
            grid_crs = aoi_gdf.estimate_utm_crs()
            print(f"Using estimated UTM CRS for AOI: {grid_crs}")
        except Exception:
            # global equal-area projection as fallback
            grid_crs = "EPSG:6933"
            print(f"Could not estimate UTM CRS for AOI, using global equal-area projection: {grid_crs}")

        aoi_proj = aoi_gdf.to_crs(grid_crs)
        shots_proj = shots_gdf.to_crs(grid_crs)

    # remove shots outside the AOI/geoJSON bounds
    aoi_union = aoi_proj.union_all() # combine every row of the geometry column into 1 vector
    shots_proj = shots_proj[shots_proj.geometry.within(aoi_union)].copy()

    # get the bounding box (smallest rectangle) of the AOI in projected coordinates
    minx, miny, maxx, maxy = aoi_proj.total_bounds

    # create the grid edges for the specified grid size
    x_edges = np.linspace(minx, maxx, grid_size + 1)
    y_edges = np.linspace(miny, maxy, grid_size + 1)

    x = shots_proj.geometry.x
    y = shots_proj.geometry.y

    # for every x y cord, find its col/row in the grid
    tile_col = np.searchsorted(x_edges, x, side="right") - 1
    tile_row = np.searchsorted(y_edges, y, side="right") - 1

    # for points that fall exactly on the max edge (they land on 10, assign them to 9)
    tile_col = np.clip(tile_col, 0, grid_size - 1)
    tile_row = np.clip(tile_row, 0, grid_size - 1)

    # add each points box to the geodataframe
    shots_proj["tile_col"] = tile_col
    shots_proj["tile_row"] = tile_row
    shots_proj["tile_id"] = [
        f"row{r:02d}_col{c:02d}" for r, c in zip(tile_row, tile_col)
    ]

    grouped = shots_proj.groupby("tile_id") # for efficiency

    grid_records = []

    for row in range(grid_size):
        for col in range(grid_size):
            # for every box in the grid do the below
            tile_id = f"row{row:02d}_col{col:02d}"

            # creates a polygon for the tile using the grid edges (used at bottom)
            tile_geom = box(
                x_edges[col],
                y_edges[row],
                x_edges[col + 1],
                y_edges[row + 1]
            )

            grid_records.append({
                "tile_id": tile_id,
                "tile_row": row,
                "tile_col": col,
                "geometry": tile_geom
            })

            if tile_id in grouped.groups:
                tile_gdf = grouped.get_group(tile_id).copy()
            else:
                print("WARNING: No points found in tile: ", tile_id)
                continue

            # Save tile back in lat/lon CRS
            tile_gdf = tile_gdf.to_crs(original_shots_crs)

            out_file = output_dir / f"{prefix}_{tile_id}.parquet"

            # save the tile a geoparquet with the id as its name
            tile_gdf.to_parquet(out_file, compression="snappy")

    # Save the tile grid itself too
    grid_gdf = gpd.GeoDataFrame(
        grid_records,
        geometry="geometry",
        crs=grid_crs
    ).to_crs(original_shots_crs)

    grid_file = output_dir / f"{prefix}_{grid_size}x{grid_size}_grid.parquet"
    grid_gdf.to_parquet(grid_file, compression="snappy")


<h3> Running the Grid Tiling Function on GEDI Points</h3>

In [ ]:
shots_gdf = gpd.read_parquet(out_parquet_file)
aoi_gdf = gpd.read_file(geojson_file_path)

tile_gedi_points_by_aoi(shots_gdf, aoi_gdf, output_dir="tiled_gedi_points", preferred_utm="EPSG:10857")

---

<h1> YOU WILL NEED TO JOIN THE GEDI DATA FILE WITH PRODES DATA BETWEEN THE PREVIOUS STEP AND THE NEXT STEP </h1>

<h4> Once you do so, place the joined CSV in the same directory as this notebook </h4>

<h1> Analyzing the GEDI + PRODES joined data </h1>

<h3> Confirm the data exists in the right location </h3>

In [ ]:
joined_csv_name = input("Enter the name of the joined CSV file (with .csv extension) that contains the data. Make sure it is in the same directory as this notebook: ")
joined_csv_path = os.path.join(curr_dir, joined_csv_name)

if not os.path.exists(joined_csv_path):
    print(f"File '{joined_csv_name}' not found in the current folder. Please check the name and location and try again.")
else:
    print(f"Joined CSV file found: {joined_csv_name}")

<h3> Create functions to extract date and land cover type from the PRODES data </h3>

Your 'source_url' row should have values like this: GEDI02_A_2019262193446_O04362_01_T00992_02_003_01_V002_subsetted.h5

In [ ]:
def get_date(url):
    # GEDI02_A_2019262193446_O04362_01_T00992_02_003_01_V002_subsetted.h5 represents day 262 of 2019
    date_str = url.split('_')[2] # 2019262193446
    year = int(date_str[:4]) # 2019
    day = int(date_str[4:7]) # 262

    date = datetime(year, 1, 1) + timedelta(days = day - 1)
    return date.strftime('%Y-%m-%d')

def label_raster_val(value):
    if pd.isna(value):
        return 'Unknown'
        
    value = int(value)
    
    if 0 <= value <= 24:
        return f'deforestation_{2000 + value}'
    
    # Residual years (forgotten to be marked as deforestation)
    elif 50 <= value <= 64:
        return f'residual_{2010 + (value - 50)}'
    
    elif value == 91:
        return 'Hydrography'
    elif value == 100:
        return 'Native Vegetation'
    elif value == 101:
        return 'Non Forest'
    
    # Unknown
    else:
        return 'Unknown'

<h3> Read in the data while applying the above functions to create new columns for date and land cover type </h3>

In [ ]:
# Get the CSV from link - Process entire file in chunks
path = joined_csv_path

chunk_size = 100000  # Process 100k rows at a time
chunks = []
row_offset = 0

print("Reading file in chunks...")
for i, chunk in enumerate(pd.read_csv(path, chunksize=chunk_size)):
    original_chunk_len = len(chunk)
    # Apply transformations to each chunk using defined functions
    for idx, row in chunk.iterrows():
        global_row = row_offset + idx + 1   # +1 if you want 1-based indexing

        try: 
            get_date(row['source_url'])
            # label_raster_val(row['RASTERVALU'])
        except Exception as e:
            print("\nERROR DETECTED")
            print(f"Chunk: {i+1}")
            print(f"CSV Row Number: {global_row}")
            print("Row contents:")
            print(row)
            print("Error:", e)
            
            raise SystemExit("Stopping execution so you can debug.")
    
    chunk['date'] = chunk['source_url'].apply(get_date)
    # chunk['land_class'] = chunk['RASTERVALU'].apply(label_raster_val)
    
    # Remove rows marked as deforestation_2007
    # chunk = chunk[chunk['land_class'] != 'deforestation_2007']
    
    chunks.append(chunk)
    print(f"  Processed chunk {i+1} ({len(chunk)} rows)")

    row_offset += original_chunk_len

# Combine all chunks
df = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows loaded: {len(df):,}")
print(df.iloc[0])